In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
import torch.nn as nn


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),                    # convert image to HxWxC tensors with floats in [0,1]
    transforms.Normalize((0.5,), (0.5,)),     # [-1, 1], cause we do (img - 0.5)/0.5 to get a centered around 0 and unit distribution
])

train_set = datasets.MNIST(
    root="./data",        
    train=True,
    download=True,
    transform=transform,
)

test_set = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=128, shuffle=False, num_workers=0)


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 500kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.53MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.68MB/s]


In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


In [12]:
class convnet(nn.Module):
    def __init__(self, in_ch, dim_out):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 32, kernel_size=3, padding=1) #(B, in_ch, 28, 28) -> (B, 32, 28, 28)
        self.act1 = nn.SiLU()
        self.pool1 = nn.MaxPool2d(2, stride=2) # (B, 32, 28, 28) -> (B, 32, 14, 14)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1) # -> (B, 64, 14, 14)
        self.act2 = nn.SiLU()
        self.pool2 = nn.MaxPool2d(2, stride=2) # -> (B, 64, 7, 7)
        self.fc1 = nn.Linear(64*7*7, 128)
        self.actfc = nn.SiLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(128, dim_out)
    def forward(self, x):
        x = self.conv1(x)
        x = self.act1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.act2(x)
        x = self.pool2(x)
        x = x.flatten(1)
        x = self.fc1(x)
        x = self.actfc(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


    

In [13]:
#test check
xt = torch.randn(128, 1, 28, 28)
model = convnet(1, 10)
out = model(xt)
print(out.shape)

torch.Size([128, 10])


In [ ]:
model = convnet(1, 10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
losses = []  
def train_loops(dataloader, model, optimizer):
    model.train()
    for _, (X, y) in enumerate(dataloader):
        X = X.to(device)
        y = y.to(device)
        loss = nn.CrossEntropyLoss()(model(X), y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
epochs = 500 
for epoch in range(epochs):
    train_loops(train_loader, model, optimizer)
    if epoch % 100 == 0:
        print(f"epoch {epoch}: loss = {losses[-1]:.6f}")
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("iteration")
plt.ylabel("loss")
plt.yscale("log")  
plt.title("Training loss")
plt.grid(True, alpha=0.3)
plt.show()

epoch 0: loss = 0.034978
